# Análisis de SBOMs y Vulnerabilidades - Organización `openclaw`

Este notebook implementa el análisis de código fuente, descubrimiento de dependencias, escaneo de vulnerabilidades y un análisis cuantitativo de la organización `openclaw`. Todo el análisis está autocontenido e invoca a **Syft** y **Grype** mediante Docker, por lo que puede ser reproducido por un tercero en cualquier entorno con Python y Docker instalados.


## 0. Prerrequisitos
Asegúrate de tener instaladas las siguientes librerías de Python. También debes tener el demonio de **Docker** instalado y corriendo en tu computadora.


In [ ]:
!pip install requests pandas matplotlib seaborn


## 1. Importación de módulos y configuración inicial


In [ ]:

import os
import subprocess
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Configuración visual para gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

ORG = "openclaw"



## 2. Descubrimiento y Descarga del Código Fuente
Obtenemos los repositorios activos de la organización usando la API pública de GitHub. Luego clonamos su código en la carpeta local `./data/repos/` para su posterior análisis.


In [ ]:

os.makedirs("data/repos", exist_ok=True)

# 1. Obtener la lista de repositorios
url_api = f"https://api.github.com/orgs/{ORG}/repos"
response = requests.get(url_api)
repos = response.json()

# Filtrar: solo queremos repos válidos y podemos imponer la misma regla de hace un mes
since_date = (datetime.now() - timedelta(days=30)).isoformat()
active_repos = [
    r for r in repos 
    if r['pushed_at'] > since_date and not r['fork']
]
print(f"Repositorios activos encontrados para descargar: {len(active_repos)}")

# 2. Descargar código fuente (.zip) de los repositorios
for repo in active_repos:
    repo_name = repo['name']
    default_branch = repo['default_branch']
    zip_url = f"https://github.com/{ORG}/{repo_name}/archive/refs/heads/{default_branch}.zip"
    
    repo_dir = f"data/repos/{repo_name}"
    zip_path = f"data/repos/{repo_name}.zip"
    
    if not os.path.exists(repo_dir):
        print(f"Descargando {repo_name}...")
        r = requests.get(zip_url, stream=True)
        if r.status_code == 200:
            with open(zip_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            # Extraer con unzip o código nativo (usamos subprocess para simplificar si unzip está disponible, o shutil natively)
            import zipfile
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall("data/repos/")
            # Renombrar la carepta extraida (-main o -master) al nombre del repositorio
            extracted_dir = f"data/repos/{repo_name}-{default_branch}"
            if os.path.exists(extracted_dir):
                os.rename(extracted_dir, repo_dir)
            os.remove(zip_path) # Limpiar zip
        else:
            print(f"Error descargando {repo_name}")
    else:
        print(f"El repositorio {repo_name} ya está descargado en {repo_dir}.")
print("Descarga completa.")



## 3. Generación de SBOMs (Syft)
Usamos el contenedor `anchore/syft:latest` para generar la boleta de materiales de software (SBOM). Esto escudriña el código fuente buscando dependencias e indirectas, emitiéndolo en formato JSON.


In [ ]:

os.makedirs("data/sboms", exist_ok=True)
cwd = os.getcwd()

print("Generando SBOMs...")
for repo_name in os.listdir("data/repos"):
    out_file = f"data/sboms/{repo_name}.json"
    repo_path = f"data/repos/{repo_name}"
    
    if os.path.isdir(repo_path) and not os.path.exists(out_file):
        print(f"Ejecutando Syft para {repo_name}...")
        # Comando Docker para Syft
        cmd = [
            "docker", "run", "--rm", 
            "-v", f"{cwd}:/workspace", 
            "anchore/syft:latest", 
            "packages", f"dir:/workspace/{repo_path}", 
            "-o", "json"
        ]
        with open(out_file, "w") as f:
            subprocess.run(cmd, stdout=f, stderr=subprocess.PIPE)
    elif os.path.exists(out_file):
        pass # print(f"SBOM para {repo_name} ya existe.")
print("Generación de SBOMs completada.")



## 4. Escaneo de Vulnerabilidades (Grype)
Ahora pasamos el SBOM generado por Grype, quien comprobará cada paquete contra las múltiples bases de datos de vulnerabilidades y vulnerabilidades expuestas (CVEs).


In [ ]:

os.makedirs("data/vulnerabilidades", exist_ok=True)

print("Escaneando vulnerabilidades...")
for repo_name in os.listdir("data/repos"):
    sbom_file = f"data/sboms/{repo_name}.json"
    vuln_file = f"data/vulnerabilidades/{repo_name}.json"
    
    if os.path.exists(sbom_file) and not os.path.exists(vuln_file):
        print(f"Ejecutando Grype para {repo_name}...")
        cmd = [
            "docker", "run", "--rm", 
            "-v", f"{cwd}:/workspace", 
            "anchore/grype:latest", 
            f"sbom:/workspace/{sbom_file}", 
            "-o", "json"
        ]
        with open(vuln_file, "w") as f:
            subprocess.run(cmd, stdout=f, stderr=subprocess.PIPE)
    elif os.path.exists(vuln_file):
        pass # print(f"Vulnerabilidades para {repo_name} ya escaneadas.")
print("Escaneo iterativo completado.")



## 5. Análisis Cuantitativo de Dependencias
Cargamos todos los SBOMs generados en un único DataFrame de Pandas para su visualización.


In [ ]:

deps_data = []
for file in os.listdir("data/sboms"):
    if file.endswith(".json"):
        repo = file.replace(".json", "")
        with open(f"data/sboms/{file}", "r") as f:
            try:
                data = json.load(f)
                artifacts = data.get("artifacts", [])
                for a in artifacts:
                    deps_data.append({
                        "repo": repo,
                        "name": a.get("name"),
                        "version": a.get("version"),
                        "type": a.get("type"),    # npm, pypi, python, etc
                        "language": a.get("language")
                    })
            except Exception as e:
                print(f"Error procesando {file}: {e}")

df_deps = pd.DataFrame(deps_data)
print(f"Se encontraron un total de {len(df_deps)} dependencias en toda la organización.")

# Muestra de datos
df_deps.head()



### 5.1 Gráfico: Ecosistemas de Paquetes más Populares


In [ ]:

plt.figure(figsize=(10, 6))
# Contar el tipo de ecosistema/librería
ecosystem_counts = df_deps['type'].value_counts()

sns.barplot(x=ecosystem_counts.values, y=ecosystem_counts.index, palette="viridis")
plt.title(f"Distribución de Ecosistemas de Paquetes en '{ORG}'")
plt.xlabel("Cantidad de dependencias encontradas")
plt.ylabel("Ecosistema (Tipo)")
plt.tight_layout()
plt.show()



## 6. Análisis Cuantitativo de Vulnerabilidades


In [ ]:

vuln_data = []
for file in os.listdir("data/vulnerabilidades"):
    if file.endswith(".json"):
        repo = file.replace(".json", "")
        with open(f"data/vulnerabilidades/{file}", "r") as f:
            try:
                data = json.load(f)
                matches = data.get("matches", [])
                for m in matches:
                    vuln = m.get("vulnerability", {})
                    artifact = m.get("artifact", {})
                    vuln_data.append({
                        "repo": repo,
                        "vulnerability_id": vuln.get("id"),
                        "severity": vuln.get("severity"),
                        "package_name": artifact.get("name"),
                        "package_version": artifact.get("version"),
                        "type": artifact.get("type")
                    })
            except Exception as e:
                print(f"Error procesando {file}: {e}")

df_vulns = pd.DataFrame(vuln_data)
if not df_vulns.empty:
    print(f"Se encontraron {len(df_vulns)} vulnerabilidades (CVEs) en total.")
    display(df_vulns.head())
else:
    print("¡No se encontraron vulnerabilidades en la organización! ¡Excelente seguridad!")



### 6.1 Gráfico: Nivel de Severidad de Vulnerabilidades


In [ ]:

import numpy as np

if not df_vulns.empty:
    plt.figure(figsize=(8, 5))
    
    # Orden lógico de severidad
    severity_order = ["Critical", "High", "Medium", "Low", "Unknown", "Negligible"]
    valid_severities = [s for s in severity_order if s in df_vulns['severity'].unique()]
    
    # Colores semánticos
    color_map = {
        "Critical": "#8b0000",
        "High": "#d32f2f",
        "Medium": "#f57c00",
        "Low": "#fbc02d",
        "Unknown": "#757575",
        "Negligible": "#bdbdbd"
    }
    
    sns.countplot(data=df_vulns, x='severity', order=valid_severities, palette=color_map)
    plt.title("Conteo de Vulnerabilidades por Nivel de Gravedad")
    plt.xlabel("Gravedad (Severity)")
    plt.ylabel("Cantidad de Vulnerabilidades")
    plt.show()



### 6.2 Top 10 Paquetes Vulnerables
Identificamos cuáles son los 10 paquetes específicos que más vulnerabilidades están aportando al ecosistema de la organización.


In [ ]:

if not df_vulns.empty:
    plt.figure(figsize=(10, 6))
    top_vuln_packages = df_vulns['package_name'].value_counts().head(10)
    
    sns.barplot(x=top_vuln_packages.values, y=top_vuln_packages.index, palette="mako")
    plt.title("Top 10 Paquetes con Mayor Cantidad de Vulnerabilidades")
    plt.xlabel("Número de vulnerabilidades reportadas")
    plt.ylabel("Nombre del Paquete")
    plt.tight_layout()
    plt.show()

